# PROCESADO DEL ARCHIVO .ttl DE AI-KG

El ttl tiene el contenido final del proyecto.  
Contiene las tripletas reificadas --> Tripletas más las propiedades.  
Para cargar el archivo en BBDD hay que extraer estas tripletas.  


Cada tripleta viene con esta estructura:

aikg:statement_110533 a aikg-ont:Statement, provo:Entity ;  
aikg-ont:hasSupport 4 ;  
aikg-ont:isInferredByTransitivity false ;  
aikg-ont:isInverse false ;  
rdf:subject aikg:learning_algorithm ;  
rdf:predicate aikg-ont:usesMethod ;  
rdf:object aikg:gradient_descent ;  
provo:wasDerivedFrom aikg:1517004310,
aikg:1973720487,
aikg:1996503769,
aikg:2085159862 ;  
provo:wasGeneratedBy aikg:DyGIE++,
aikg:OpenIE,
aikg:pipeline_V1.2 .


Cada prefijo es una abreviatura de una URL (tambien hay que procesaarlo):

@prefix aikg: <http://scholkg.kmi.open.ac.uk/aikg/resource/> .  
@prefix aikg-ont: <http://scholkg.kmi.open.ac.uk/aikg/ontology#> .  
@prefix cso: <https://cso.kmi.open.ac.uk/topics/> .  
@prefix owl: <http://www.w3.org/2002/07/owl#> .  
@prefix provo: <http://www.w3.org/ns/prov#> .  
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .  
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .  
@prefix xml: <http://www.w3.org/XML/1998/namespace> .  
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .  


# 0.    IMPORTS

In [ ]:
from rdflib import Graph, Namespace
from rdflib.namespace import RDF, split_uri
import csv
from tqdm import tqdm
import pandas as pd

# 1. REDUCCION DEL ARCHIVO ORIGINAL

In [2]:
ruta = "aikg/"

in_name = "aikg.ttl"
out_name = "aikg_sample.ttl"

input_file = f"{ruta}{in_name}"
sample_file = f"{ruta}{out_name}"

In [ ]:
max_triples = 10000
count = 0

with open(input_file, "r", encoding="utf-8") as f_in, \
     open(sample_file, "w", encoding="utf-8") as f_out:
    
    buffer = ""
    
    for line in f_in:
        buffer += line
        
        if line.strip().endswith("."):
            f_out.write(buffer)
            buffer = ""
            count += 1
        
        if count >= max_triples:
            break

Si se intentan ver las tripletas directamente se observa que estan desordenadas y ademas contienen los prefijos explicitamente:

In [3]:
g = Graph()

g.parse(sample_file, format="turtle")

print(len(g))  # número de tripletas

116469


In [7]:
# for s, p, o in list(g)[:10]:
for s, p, o in list(g)[:10]:
    print(s, p, o)

http://scholkg.kmi.open.ac.uk/aikg/resource/statement_1001116 http://www.w3.org/1999/02/22-rdf-syntax-ns#object http://scholkg.kmi.open.ac.uk/aikg/resource/disparity
http://scholkg.kmi.open.ac.uk/aikg/resource/statement_1005598 http://scholkg.kmi.open.ac.uk/aikg/ontology#isInverse true
http://scholkg.kmi.open.ac.uk/aikg/resource/statement_1005630 http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/1999/02/22-rdf-syntax-ns#Statement
http://scholkg.kmi.open.ac.uk/aikg/resource/statement_1005660 http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/1999/02/22-rdf-syntax-ns#Statement
http://scholkg.kmi.open.ac.uk/aikg/resource/statement_1005758 http://www.w3.org/ns/prov#wasDerivedFrom http://scholkg.kmi.open.ac.uk/aikg/resource/2133656631
http://scholkg.kmi.open.ac.uk/aikg/resource/statement_1004541 http://www.w3.org/ns/prov#wasDerivedFrom http://scholkg.kmi.open.ac.uk/aikg/resource/2141541378
http://scholkg.kmi.open.ac.uk/aikg/resource/statement_1007036 http://sc

# 2. EXTRACCION DE TRIPLETAS

## 2.0 Explicacion codigo

🔹 Graph()  
Contenedor de triples RDF  
🔹 parse()  
Lee archivo TTL → lo convierte a triples  
🔹 Namespace()  
Define prefijos reutilizables  
Permite usar RDF.type en vez de URL completa  
🔹 g.subjects(p, o)  
Devuelve sujetos que cumplen (s, p, o)  
🔹 g.value(s, p)  
Devuelve el objeto de (s, p, o)  

Graph:  
Es la estructura principal de rdflib  
Representa todo el grafo RDF en memoria (una colección de triples (s, p, o))

parse():  
Carga el archivo TTL dentro del grafo:  
Lee el archivo  
Interpreta los prefijos  
Convierte todo a triples RDF internos  

Namespace:  
Es una forma de trabajar con URIs sin escribirlas completas  
Permite usar:  
RDF.type  
RDF.subject  
RDF.predicate  
RDF.object  
En vez de: http://www.w3.org/1999/02/22-rdf-syntax-ns#type

RDF.type  
representa:  
rdf:type  
que significa: “este nodo es de tipo X”

g.subjects():  
g.subjects(predicado, objeto) Devuelve todos los sujetos que cumplen: (sujeto, predicado, objeto).  

g.subjects(RDF.type, None):   
dame todos los nodos que tienen:
  (X, rdf:type, algo) 

g.value():  
g.value(sujeto, predicado)   Devuelve el objeto del triple:  (sujeto, predicado, objeto)

g.value(stmt, RDF.subject):  
significa: (stmt, rdf:subject, ?)  
devuelve:  learning_algorithm

pred = g.value(stmt, RDF.predicate)  
busca: (stmt, rdf:predicate, ?)  
devuelve: usesMethod  

In [9]:
g = Graph()
g.parse(sample_file, format="turtle")

RDF = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#")

triples = []

for stmt in g.subjects(RDF.type, None):
    subj = g.value(stmt, RDF.subject)
    pred = g.value(stmt, RDF.predicate)
    obj  = g.value(stmt, RDF.object)
    
    if subj and pred and obj:
        triples.append((str(subj), str(pred), str(obj)))

print(triples[:5])

[('http://scholkg.kmi.open.ac.uk/aikg/resource/recommender_system', 'http://scholkg.kmi.open.ac.uk/aikg/ontology#supportsOtherEntity', 'http://scholkg.kmi.open.ac.uk/aikg/resource/information_overload'), ('http://scholkg.kmi.open.ac.uk/aikg/resource/knowledge_sharing', 'http://scholkg.kmi.open.ac.uk/aikg/ontology#taskSupportedBy', 'http://scholkg.kmi.open.ac.uk/aikg/resource/km'), ('http://scholkg.kmi.open.ac.uk/aikg/resource/adaboost', 'http://scholkg.kmi.open.ac.uk/aikg/ontology#methodUsedBy', 'http://scholkg.kmi.open.ac.uk/aikg/resource/boosting'), ('http://scholkg.kmi.open.ac.uk/aikg/resource/object_recognition', 'http://scholkg.kmi.open.ac.uk/aikg/ontology#otherEntityUsedBy', 'http://scholkg.kmi.open.ac.uk/aikg/resource/image_segmentation'), ('http://scholkg.kmi.open.ac.uk/aikg/resource/semantic_network', 'http://scholkg.kmi.open.ac.uk/aikg/ontology#methodUsedBy', 'http://scholkg.kmi.open.ac.uk/aikg/resource/knowledge_representation')]


In [13]:
len(triples)

24972

Mejora importante

Ahora mismo estás cogiendo todos los nodos con rdf:type, pero podrías filtrar solo los statements (solo nodos de tipo Statement):

AIKG = Namespace("http://scholkg.kmi.open.ac.uk/aikg/ontology#")  

for stmt in g.subjects(RDF.type, AIKG.Statement):

## 2.1 Limpieza de tripletas dejando prefijos

Con las tripletas ya extraidas hay que quitar los prefijos para dejar las tripletas limpias.

Ahora las tripletas tienen este formato:

('http://scholkg.kmi.open.ac.uk/aikg/resource/recommender_system', 'http://scholkg.kmi.open.ac.uk/aikg/ontology#supportsOtherEntity', 'http://scholkg.kmi.open.ac.uk/aikg/resource/information_overload')

g.namespace_manager: gestiona los prefijos del grafo

conoce todos los @prefix del TTL  
sabe mapear URI ↔ prefijo

In [18]:
g = Graph()
g.parse(sample_file, format="turtle")

RDF = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#")

triples = []

for stmt in g.subjects(RDF.type, None):
    subj = g.value(stmt, RDF.subject)
    pred = g.value(stmt, RDF.predicate)
    obj  = g.value(stmt, RDF.object)
    
    if subj and pred and obj:
        #LIMPIEZA DE URIs ANTES DE GUARDAR
        subj_qname = g.namespace_manager.normalizeUri(subj)
        pred_qname = g.namespace_manager.normalizeUri(pred)
        obj_qname  = g.namespace_manager.normalizeUri(obj)
        
        triples.append((subj_qname, pred_qname, obj_qname))

print(triples[:2])

[('aikg:recommender_system', 'aikg-ont:supportsOtherEntity', 'aikg:information_overload'), ('aikg:knowledge_sharing', 'aikg-ont:taskSupportedBy', 'aikg:km')]


In [19]:
for t in triples[:10]:
    print(t)

('aikg:recommender_system', 'aikg-ont:supportsOtherEntity', 'aikg:information_overload')
('aikg:knowledge_sharing', 'aikg-ont:taskSupportedBy', 'aikg:km')
('aikg:adaboost', 'aikg-ont:methodUsedBy', 'aikg:boosting')
('aikg:object_recognition', 'aikg-ont:otherEntityUsedBy', 'aikg:image_segmentation')
('aikg:semantic_network', 'aikg-ont:methodUsedBy', 'aikg:knowledge_representation')
('aikg:execution_time', 'aikg-ont:metricImprovedBy', 'aikg:register_allocation')
('aikg:peer_to_peer_framework', 'skos:narrower', 'aikg:business_intelligence_network')
('aikg:decision_making_process', 'aikg-ont:usesMethod', 'aikg:collaborative_bi')
('aikg:collaborative_bi', 'aikg-ont:methodUsedBy', 'aikg:decision_making_process')
('aikg:model_oriented_formalism', 'aikg-ont:usesOtherEntity', 'aikg:safety_constraint')


## 2.2 Limpieza de tripletas SIN prefijos

MANUAL 

In [ ]:
# "http://scholkg.kmi.open.ac.uk/aikg/resource/recommender_system".split("/")[-1].split("#")[-1]

CON LIBRERIA rdflib

In [ ]:
g = Graph()
# g.parse(sample_file, format="turtle")
g.parse(input_file, format="turtle")

RDF = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#")
# Namespace AIKG
AIKG = Namespace("http://scholkg.kmi.open.ac.uk/aikg/ontology#")

triples = []

for stmt in tqdm(g.subjects(RDF.type, None)):
# for stmt in g.subjects(RDF.type, AIKG.Statement): # Para recorrer SOLO Statements
    subj = g.value(stmt, RDF.subject)
    pred = g.value(stmt, RDF.predicate)
    obj  = g.value(stmt, RDF.object)
    
    if subj and pred and obj:
        #LIMPIEZA DE URIs ANTES DE GUARDAR
        '''
        # DEJANDO EL PREFIJO
        subj_qname = g.namespace_manager.normalizeUri(subj)
        pred_qname = g.namespace_manager.normalizeUri(pred)
        obj_qname  = g.namespace_manager.normalizeUri(obj)
        '''
        # SIN PREFIJO
        subj_clean = split_uri(subj)[1]
        pred_clean = split_uri(pred)[1]
        obj_clean  = split_uri(obj)[1]

        triples.append((subj_clean, pred_clean, obj_clean))

print(triples[:2])

In [ ]:
print(len(triples))

In [5]:
for t in triples[:10]:
    print(t)

('recommender_system', 'supportsOtherEntity', 'information_overload')
('knowledge_sharing', 'taskSupportedBy', 'km')
('adaboost', 'methodUsedBy', 'boosting')
('object_recognition', 'otherEntityUsedBy', 'image_segmentation')
('semantic_network', 'methodUsedBy', 'knowledge_representation')
('execution_time', 'metricImprovedBy', 'register_allocation')
('peer_to_peer_framework', 'narrower', 'business_intelligence_network')
('decision_making_process', 'usesMethod', 'collaborative_bi')
('collaborative_bi', 'methodUsedBy', 'decision_making_process')
('model_oriented_formalism', 'usesOtherEntity', 'safety_constraint')


# 3. Convertir relaciones a estilo Neo4j

In [ ]:
import inflection


In [ ]:
def to_neo4j_rel(rel):
    return inflection.underscore(rel).upper()

In [ ]:
triples = [
    (s, to_neo4j_rel(p), o)
    for s, p, o in triples
]

In [14]:
for t in triples[:10]:
    print(t)

('recommender_system', 'SUPPORTS_OTHER_ENTITY', 'information_overload')
('knowledge_sharing', 'TASK_SUPPORTED_BY', 'km')
('adaboost', 'METHOD_USED_BY', 'boosting')
('object_recognition', 'OTHER_ENTITY_USED_BY', 'image_segmentation')
('semantic_network', 'METHOD_USED_BY', 'knowledge_representation')
('execution_time', 'METRIC_IMPROVED_BY', 'register_allocation')
('peer_to_peer_framework', 'NARROWER', 'business_intelligence_network')
('decision_making_process', 'USES_METHOD', 'collaborative_bi')
('collaborative_bi', 'METHOD_USED_BY', 'decision_making_process')
('model_oriented_formalism', 'USES_OTHER_ENTITY', 'safety_constraint')


# 4. EXPORTAR A CSV

## 4.1 PYTHON

In [ ]:
# csv_name = "aikg_sample.csv"
csv_name = "aikg.csv"
csv_file = f"{ruta}{csv_name}"

In [ ]:
with open(csv_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    # Cabecera
    writer.writerow(["subject", "predicate", "object"])

    # Datos
    for s, p, o in triples:
        writer.writerow([s, p, o])

## 4.2 PANDAS

In [ ]:
df = pd.DataFrame(triples, columns=["subject", "predicate", "object"])
df.to_csv("triples.csv", index=False)